# Milestone 3 - Modelacao Preditiva e Contexto

Grupo 2

João Costa | 113564

Sílvia Ribeiro | 129428

Vitória Rodrigues | 130557

Este notebook implementa a fase de **modelacao preditiva com sinais de contexto** do projeto SIMIFTA, utilizando o ficheiro `df_preprocessed_eda.csv` na pasta do Milestone 3.

O fluxo segue os requisitos definidos:

- filtragem dos 4 POIs principais
- integracao de sinais de montante selecionados
- tratamento de gaps no sinal de chegadas
- integracao de engajamento digital com protecao contra vi?s temporal
- rejeicao justificada da tabela de pagamentos
- split temporal anti-leakage
- comparacao entre Regressao Linear e CatBoostRegressor
- GridSearchCV com early stopping para o CatBoost
- logica de gestao com `Crowding Factor` e alerta GSTC A8
- validacao do objetivo de negocio: **MAPE < 20%**

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Caminhos, ficheiros e criterio de sucesso

O notebook procura automaticamente a pasta central `dataset` a partir da diretoria atual de execução do Jupyter, centralizando a leitura dos dados limpos e dos sinais externos de contexto (Arrivals, Engajamento Digital e Pagamentos).

In [ ]:
START_DIR = Path.cwd().resolve()
print(f"Diretoria atual de execução: {START_DIR}")

DATA_DIR = None
for root in [START_DIR, *START_DIR.parents]:
    candidate = root / "dataset"
    if candidate.exists() and candidate.is_dir():
        DATA_DIR = candidate.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta central 'dataset'. "
        "Certifica-te de que manténs a pasta 'dataset' na raiz do projeto."
    )

print(f"Pasta unificada de dados encontrada em: {DATA_DIR}")

MAIN_DATA_PATH = DATA_DIR / 'df_preprocessed_eda.csv'
ARRIVALS_PATH = DATA_DIR / 'fact_visitor_arrivals_departures.csv'
DIGITAL_PATH = DATA_DIR / 'fact_digital_engagement.csv'
PAYMENTS_PATH = DATA_DIR / 'fact_payment_statistics.csv'
TARGET_BUSINESS_MAPE = 0.20

for path in [MAIN_DATA_PATH, ARRIVALS_PATH, DIGITAL_PATH, PAYMENTS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Ficheiro necessário não encontrado: {path}")

print(f"Dataset principal: {MAIN_DATA_PATH}")
print(f"Meta de negocio: MAPE < {TARGET_BUSINESS_MAPE:.0%}")

## 2. Carregamento do dataset principal e filtro dos POIs

Os quatro POIs considerados sao:

- Sete Cidades - Ponte entre Lagoas
- Vista do Rei - Miradouro
- Lagoa do Canario
- Caldeiras das Furnas

In [ ]:
TARGET_POIS = {
    9246: 'Sete Cidades - Ponte entre Lagoas',
    10026: 'Vista do Rei - Miradouro',
    7175: 'Lagoa do Canario',
    6967: 'Caldeiras das Furnas',
}

main_df = pd.read_csv(MAIN_DATA_PATH)
main_df['movement_date'] = pd.to_datetime(main_df['movement_date'], errors='coerce')
main_df = main_df[main_df['geography_key'].isin(TARGET_POIS.keys())].copy()
main_df['poi_label'] = main_df['geography_key'].map(TARGET_POIS)
main_df = main_df.sort_values(['geography_key', 'movement_date']).reset_index(drop=True)

print(f"Shape do dataset principal apos filtro: {main_df.shape}")
display(main_df[['movement_date', 'geography_key', 'poi_name', 'poi_label', 'total_incoming']].head(12))
print('Cobertura temporal:', main_df['movement_date'].min().date(), 'a', main_df['movement_date'].max().date())

## 3. Capacidade maxima historica e referencia operacional

A capacidade maxima de cada POI e definida como o **pico historico de `total_incoming`**. Esta referencia sera usada na logica de `Crowding Factor` e no alerta de risco de sobrelotacao.

In [ ]:
capacity_df = (
    main_df.groupby(['geography_key', 'poi_label'], dropna=False)['total_incoming']
           .max()
           .rename('poi_capacity_max')
           .reset_index()
           .sort_values('poi_capacity_max', ascending=False)
)

main_df = main_df.merge(
    capacity_df[['geography_key', 'poi_capacity_max']],
    on='geography_key',
    how='left',
    validate='m:1'
)
main_df['observed_occupancy_rate'] = main_df['total_incoming'] / main_df['poi_capacity_max']

display(capacity_df)

## 4. Sinal de montante 1: chegadas a regiao (ARRIVAL)

A tabela `FACT_VISITOR_ARRIVALS_DEPARTURES` e filtrada para `flow_type == 'ARRIVAL'`. A metrica de interesse e o volume diario de passageiros desembarcados (`passenger_count`) agregado por data.

In [ ]:
arrivals_df = pd.read_csv(ARRIVALS_PATH)
arrivals_df = arrivals_df[arrivals_df['flow_type'].astype(str).str.upper() == 'ARRIVAL'].copy()
arrivals_df['arrival_date'] = pd.to_datetime(arrivals_df['arrival_date'], errors='coerce')

arrivals_daily = (
    arrivals_df.dropna(subset=['arrival_date'])
               .groupby('arrival_date', as_index=False)['passenger_count']
               .sum()
               .rename(columns={
                   'arrival_date': 'movement_date',
                   'passenger_count': 'daily_arrivals_passenger_count'
               })
               .sort_values('movement_date')
)

print(f"Shape do agregado diario de arrivals: {arrivals_daily.shape}")
display(arrivals_daily.tail(12))

## 5. Tratamento de gaps no sinal de chegadas

O enunciado exige o tratamento do intervalo **05/10/2025 a 25/10/2025**. Para esse fim, aplicamos uma **imputacao por mediana sazonal**, usando a mediana do mesmo dia da semana observada nas semanas vizinhas.

A estrategia implementada e:

1. reindexacao diaria da serie de arrivals
2. imputacao apenas no intervalo solicitado
3. procura de valores do mesmo dia da semana em janelas anteriores e posteriores
4. fallback para a mediana global do mesmo dia da semana, se necessario

In [ ]:
arrivals_daily = (
    arrivals_daily.set_index('movement_date')
                 .reindex(pd.date_range(arrivals_daily['movement_date'].min(), arrivals_daily['movement_date'].max(), freq='D'))
                 .rename_axis('movement_date')
                 .reset_index()
)


def seasonal_weekday_impute(df, target_col, start_date, end_date, window_weeks=3):
    df = df.copy()
    df['day_of_week_tmp'] = df['movement_date'].dt.dayofweek
    target_mask = (df['movement_date'] >= pd.Timestamp(start_date)) & (df['movement_date'] <= pd.Timestamp(end_date))

    for idx, row in df.loc[target_mask].iterrows():
        if pd.notna(row[target_col]):
            continue

        current_date = row['movement_date']
        weekday = row['day_of_week_tmp']

        neighbour_mask = (
            (df['day_of_week_tmp'] == weekday) & (
                ((df['movement_date'] >= current_date - pd.Timedelta(days=7 * window_weeks)) & (df['movement_date'] <= current_date - pd.Timedelta(days=7))) |
                ((df['movement_date'] >= current_date + pd.Timedelta(days=7)) & (df['movement_date'] <= current_date + pd.Timedelta(days=7 * window_weeks)))
            )
        )

        neighbour_values = df.loc[neighbour_mask, target_col].dropna()

        if neighbour_values.empty:
            neighbour_values = df.loc[(df['day_of_week_tmp'] == weekday) & df[target_col].notna(), target_col]

        if neighbour_values.empty:
            neighbour_values = df[target_col].dropna()

        if not neighbour_values.empty:
            df.at[idx, target_col] = float(neighbour_values.median())

    return df.drop(columns='day_of_week_tmp')


arrivals_daily = seasonal_weekday_impute(
    arrivals_daily,
    target_col='daily_arrivals_passenger_count',
    start_date='2025-10-05',
    end_date='2025-10-25'
)

arrivals_daily['arrivals_imputed_flag'] = arrivals_daily['daily_arrivals_passenger_count'].notna().astype(int)

display(
    arrivals_daily[(arrivals_daily['movement_date'] >= pd.Timestamp('2025-09-24')) & (arrivals_daily['movement_date'] <= pd.Timestamp('2025-10-25'))]
)

In [ ]:
model_df = main_df.merge(arrivals_daily, on='movement_date', how='left')
model_df['has_arrival_signal'] = model_df['daily_arrivals_passenger_count'].notna().astype(int)

print('Cobertura do sinal de arrivals nas datas do dataset principal:')
display(
    model_df[['movement_date', 'daily_arrivals_passenger_count', 'arrivals_imputed_flag', 'has_arrival_signal']]
    .drop_duplicates()
    .sort_values('movement_date')
)

## 6. Sinal de montante 2: engajamento digital

A tabela `FACT_DIGITAL_ENGAGEMENT` e integrada com foco em:

- `search_volume`
- `sentiment_score`

Como estes dados so existem a partir de **01/10/2025**, os dias anteriores recebem valores de preenchimento controlados:

- `search_volume = 0`
- `sentiment_score = media historica disponivel` (ou 0 se nao existir media)
- `has_digital_signal = 0` antes do primeiro dia com disponibilidade estrutural

Esta estrategia evita que o modelo interprete a ausencia estrutural de dados como um sinal semantico forte.

In [ ]:
digital_df = pd.read_csv(DIGITAL_PATH, usecols=['engagement_date', 'search_volume', 'sentiment_score'])
digital_df['engagement_date'] = pd.to_datetime(digital_df['engagement_date'], errors='coerce')

digital_daily = (
    digital_df.groupby('engagement_date', as_index=False)
              .agg(
                  search_volume=('search_volume', 'sum'),
                  sentiment_score=('sentiment_score', 'mean')
              )
              .rename(columns={'engagement_date': 'movement_date'})
              .sort_values('movement_date')
)

first_digital_signal_date = digital_daily['movement_date'].min()
sentiment_fill_value = digital_daily['sentiment_score'].mean(skipna=True)
if pd.isna(sentiment_fill_value):
    sentiment_fill_value = 0.0

model_df = model_df.merge(digital_daily, on='movement_date', how='left')
model_df['has_digital_signal'] = (model_df['movement_date'] >= first_digital_signal_date).astype(int)
model_df.loc[model_df['movement_date'] < first_digital_signal_date, 'search_volume'] = 0.0
model_df.loc[model_df['movement_date'] < first_digital_signal_date, 'sentiment_score'] = sentiment_fill_value
model_df['search_volume'] = model_df['search_volume'].fillna(0.0)
model_df['sentiment_score'] = model_df['sentiment_score'].fillna(sentiment_fill_value)

print(f"Primeira data estrutural do sinal digital: {first_digital_signal_date.date()}")
print(f"Valor de preenchimento usado para sentiment_score: {sentiment_fill_value:.4f}")
display(
    model_df[['movement_date', 'search_volume', 'sentiment_score', 'has_digital_signal']]
    .drop_duplicates()
    .sort_values('movement_date')
)

## 7. Rejeicao da tabela de pagamentos

A tabela de pagamentos existe no projeto, mas **nao e utilizada** neste notebook. A justificacao metodologica e simples:

- a granularidade observada e agregada e inadequada para previsao **diaria** de picos
- o objetivo do modelo e prever variacoes de curto prazo entre 01/10 e 09/10
- um sinal financeiro mais agregado poderia introduzir ruido ou leakage sem contribuir para a previsao operacional de crowding

In [ ]:
payments_preview = pd.read_csv(PAYMENTS_PATH, nrows=10)
print('Colunas da tabela de pagamentos:')
print(payments_preview.columns.tolist())
print('\nJustificacao de exclusao: a tabela de pagamentos nao entra no modelo porque a sua estrutura agregada nao e compativel com a previsao diaria de picos de afluencia.')


## 8. Engenharia de caracteristicas

Mantemos as variaveis pedidas e acrescentamos lags de afluencia e de arrivals para captar dependencia temporal:

- `weather_temperature_2m_mean`
- `weather_precipitation_sum`
- `is_weekend`
- `is_holiday`
- `daily_arrivals_passenger_count`
- `search_volume`
- `sentiment_score`
- `has_digital_signal`
- `total_incoming_lag_1`
- `total_incoming_lag_2`
- `total_incoming_roll3_mean`
- `arrivals_passenger_count_lag_1`

In [ ]:
model_df = model_df.sort_values(['geography_key', 'movement_date']).reset_index(drop=True)
model_df['is_weekend'] = model_df['is_weekend'].astype(int)
model_df['is_holiday'] = model_df['is_holiday'].astype(int)

for lag in [1, 2]:
    model_df[f'total_incoming_lag_{lag}'] = model_df.groupby('geography_key')['total_incoming'].shift(lag)

model_df['total_incoming_roll3_mean'] = (
    model_df.groupby('geography_key')['total_incoming']
            .shift(1)
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
)

model_df['arrivals_passenger_count_lag_1'] = model_df.groupby('geography_key')['daily_arrivals_passenger_count'].shift(1)

feature_cols = [
    'poi_label',
    'day_of_week',
    'is_weekend',
    'is_holiday',
    'weather_temperature_2m_mean',
    'weather_precipitation_sum',
    'daily_arrivals_passenger_count',
    'arrivals_passenger_count_lag_1',
    'search_volume',
    'sentiment_score',
    'has_digital_signal',
    'total_incoming_lag_1',
    'total_incoming_lag_2',
    'total_incoming_roll3_mean',
]

display(model_df[['movement_date', 'poi_label', 'total_incoming'] + feature_cols[1:]].head(14))

## 9. Split temporal anti-leakage

A divisao temporal e manual e respeita a logica operacional do problema:

- **Treino:** ate 30/09/2025
- **Teste:** de 01/10/2025 a 09/10/2025

Como o sinal digital so passa a existir estruturalmente a partir de 01/10/2025, esta janela de teste e particularmente importante para validar o contexto adicional sem leakage.

Para o CatBoost, a parte de treino e ainda dividida em:

- **Subtreino:** ate 28/09/2025
- **Validacao temporal interna:** 29/09/2025 a 30/09/2025

In [ ]:
train_df = model_df[model_df['movement_date'] <= pd.Timestamp('2025-09-30')].copy()
test_df = model_df[(model_df['movement_date'] >= pd.Timestamp('2025-10-01')) & (model_df['movement_date'] <= pd.Timestamp('2025-10-09'))].copy()

train_core_df = train_df[train_df['movement_date'] <= pd.Timestamp('2025-09-28')].copy()
val_df = train_df[(train_df['movement_date'] >= pd.Timestamp('2025-09-29')) & (train_df['movement_date'] <= pd.Timestamp('2025-09-30'))].copy()

print('Treino completo:', train_df.shape)
print('Subtreino:', train_core_df.shape)
print('Validacao interna:', val_df.shape)
print('Teste:', test_df.shape)

## 10. Matrizes de treino e funcoes auxiliares

In [ ]:
X_train = train_df[feature_cols].copy()
y_train = train_df['total_incoming'].copy()

X_train_core = train_core_df[feature_cols].copy()
y_train_core = train_core_df['total_incoming'].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df['total_incoming'].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['total_incoming'].copy()

numeric_features = [col for col in feature_cols if col != 'poi_label']
categorical_features = ['poi_label']


def regression_metrics(y_true, y_pred):
    return {
        'MAPE': mean_absolute_percentage_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'MAE': mean_absolute_error(y_true, y_pred),
    }


def clip_non_negative(values):
    return np.clip(values, a_min=0, a_max=None)

## 11. Baseline: Regressao Linear

In [ ]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

linear_model = Pipeline([
    ('preprocessor', linear_preprocessor),
    ('model', LinearRegression()),
])

linear_model.fit(X_train, y_train)
linear_pred_test = clip_non_negative(linear_model.predict(X_test))
linear_metrics = regression_metrics(y_test, linear_pred_test)

print('Metricas da Regressao Linear:')
for metric_name, metric_value in linear_metrics.items():
    if metric_name == 'MAPE':
        print(f'- {metric_name}: {metric_value:.2%}')
    else:
        print(f'- {metric_name}: {metric_value:.3f}')

## 12. Modelo avancado: CatBoostRegressor com GridSearchCV e Early Stopping

A otimizacao usa `GridSearchCV` com `PredefinedSplit`, preservando a ordem temporal:

- a grelha otimiza `depth` e `learning_rate`
- o scoring usado e o **MAPE negativo**
- o `eval_set` para early stopping e a janela de validacao interna

In [ ]:
# Preparacao para GridSearchCV temporal com validacao fixa.
X_grid = pd.concat([X_train_core, X_val], axis=0).reset_index(drop=True)
y_grid = pd.concat([y_train_core, y_val], axis=0).reset_index(drop=True)
predefined_fold = [-1] * len(X_train_core) + [0] * len(X_val)
predefined_split = PredefinedSplit(test_fold=predefined_fold)

median_fill_values = X_grid[numeric_features].median(numeric_only=True)

for frame in [X_grid, X_train_core, X_val, X_train, X_test]:
    frame[numeric_features] = frame[numeric_features].fillna(median_fill_values)
    frame['poi_label'] = frame['poi_label'].fillna('POI_desconhecido')

catboost_base = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='MAPE',
    random_seed=42,
    verbose=False,
    iterations=300,
)

catboost_grid = {
    'depth': [3, 5],
    'learning_rate': [0.03, 0.1],
}

catboost_search = GridSearchCV(
    estimator=catboost_base,
    param_grid=catboost_grid,
    scoring='neg_mean_absolute_percentage_error',
    cv=predefined_split,
    refit=False,
    n_jobs=1,
)

catboost_search.fit(
    X_grid,
    y_grid,
    cat_features=['poi_label'],
    eval_set=(X_val, y_val),
    early_stopping_rounds=20,
    verbose=False,
)

catboost_search_results = pd.DataFrame(catboost_search.cv_results_)
catboost_search_results['mean_test_mape'] = -catboost_search_results['mean_test_score']
display(
    catboost_search_results[[
        'param_depth',
        'param_learning_rate',
        'mean_test_mape',
        'rank_test_score'
    ]].sort_values('rank_test_score')
)

best_catboost_params = catboost_search.best_params_
print('Melhores hiperparametros do GridSearchCV:')
print(best_catboost_params)
print(f"Melhor MAPE de validacao interna: {-catboost_search.best_score_:.2%}")

In [ ]:
catboost_model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='MAPE',
    random_seed=42,
    verbose=False,
    iterations=300,
    **best_catboost_params,
)

catboost_model.fit(
    X_train_core,
    y_train_core,
    cat_features=['poi_label'],
    eval_set=(X_val, y_val),
    early_stopping_rounds=20,
    use_best_model=True,
    verbose=False,
)

catboost_pred_test = clip_non_negative(catboost_model.predict(X_test))
catboost_metrics = regression_metrics(y_test, catboost_pred_test)

print('Metricas do CatBoostRegressor:')
for metric_name, metric_value in catboost_metrics.items():
    if metric_name == 'MAPE':
        print(f'- {metric_name}: {metric_value:.2%}')
    else:
        print(f'- {metric_name}: {metric_value:.3f}')

## 13. Comparacao entre modelos

In [ ]:
comparison_df = pd.DataFrame([
    {'Model': 'Regressao Linear', **linear_metrics},
    {'Model': 'CatBoostRegressor', **catboost_metrics},
])
comparison_df['Business_Goal_Met'] = comparison_df['MAPE'] < TARGET_BUSINESS_MAPE
comparison_df = comparison_df.sort_values('MAPE').reset_index(drop=True)
display(comparison_df)

best_model_name = comparison_df.iloc[0]['Model']
print(f"Melhor modelo segundo o MAPE: {best_model_name}")

## 14. Crowding Factor e alerta GSTC A8

A previsao do melhor modelo e convertida para suporte a decisao:

- `predicted_occupancy_rate`
- `crowding_factor`
- `gstc_a8_alert`

Regra de alerta:

- se a previsao exceder **80% da capacidade historica**, o estado devolvido e **Risco de Sobrelotacao**

In [ ]:
def crowding_factor_from_rate(rate):
    if rate <= 0.50:
        return 'Baixo'
    if rate <= 0.80:
        return 'Moderado'
    if rate <= 1.00:
        return 'Elevado'
    return 'Critico'


def gstc_a8_alert_from_rate(rate):
    if rate > 0.80:
        return 'Risco de Sobrelotacao'
    return 'Sem alerta'

prediction_df = test_df[[
    'movement_date',
    'geography_key',
    'poi_label',
    'total_incoming',
    'poi_capacity_max',
]].copy()

prediction_df['pred_linear'] = linear_pred_test
prediction_df['pred_catboost'] = catboost_pred_test
prediction_df['predicted_occupancy_rate_catboost'] = prediction_df['pred_catboost'] / prediction_df['poi_capacity_max']
prediction_df['crowding_factor_catboost'] = prediction_df['predicted_occupancy_rate_catboost'].apply(crowding_factor_from_rate)
prediction_df['gstc_a8_alert'] = prediction_df['predicted_occupancy_rate_catboost'].apply(gstc_a8_alert_from_rate)

display(prediction_df.sort_values(['poi_label', 'movement_date']))

## 15. Graficos: afluencia real vs prevista e capacidade de carga

In [ ]:
def plot_predictions_by_poi(pred_df, prediction_column, title_prefix):
    poi_order = pred_df['poi_label'].drop_duplicates().tolist()
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
    axes = axes.flatten()

    for ax, poi_label in zip(axes, poi_order):
        poi_data = pred_df[pred_df['poi_label'] == poi_label].sort_values('movement_date')
        capacity_value = poi_data['poi_capacity_max'].iloc[0]

        ax.plot(poi_data['movement_date'], poi_data['total_incoming'], marker='o', label='Real', color='#1D3557')
        ax.plot(poi_data['movement_date'], poi_data[prediction_column], marker='o', label='Prevista', color='#E76F51')
        ax.axhline(capacity_value, linestyle='--', linewidth=2, color='#2A9D8F', label='Capacidade historica')
        ax.set_title(f'{title_prefix} - {poi_label}')
        ax.set_xlabel('Data')
        ax.set_ylabel('Afluencia')
        ax.tick_params(axis='x', rotation=30)
        ax.legend()

    plt.tight_layout()
    plt.show()

plot_predictions_by_poi(prediction_df, 'pred_linear', 'Regressao Linear')
plot_predictions_by_poi(prediction_df, 'pred_catboost', 'CatBoost')

## 16. Validacao do objetivo de negocio

O notebook deve confirmar explicitamente se o CatBoost atingiu o alvo:

- **MAPE < 20%**

In [ ]:
catboost_goal_met = bool(catboost_metrics['MAPE'] < TARGET_BUSINESS_MAPE)

print('Resumo final do CatBoost:')
print(f"- MAPE: {catboost_metrics['MAPE']:.2%}")
print(f"- RMSE: {catboost_metrics['RMSE']:.3f}")
print(f"- MAE: {catboost_metrics['MAE']:.3f}")
print(f"- Objetivo de negocio atingido? {catboost_goal_met}")
print(f"- Numero de alertas GSTC A8: {(prediction_df['gstc_a8_alert'] == 'Risco de Sobrelotacao').sum()}")

if catboost_goal_met:
    print(
        "\nConclusao: o CatBoost atingiu o objetivo de negocio de MAPE inferior a 20%. "
        "A modelacao preditiva com contexto pode ser considerada valida para suporte operacional."
    )
else:
    print(
        "\nConclusao: o CatBoost nao atingiu o objetivo de negocio de MAPE inferior a 20%. "
        "O notebook continua util por estruturar o pipeline completo de contexto, anti-leakage e apoio a decisao GSTC, mas o modelo necessita de melhoria adicional."
    )


## 17. Exportacao dos resultados

Criamos uma subpasta organizada dentro do nosso dataset central para os resultados do modelo

In [ ]:
OUTPUT_DIR = DATA_DIR / 'export_model'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comparison_output_path = OUTPUT_DIR / 'modelacao_contexto_metrics.csv'
predictions_output_path = OUTPUT_DIR / 'modelacao_contexto_predictions.csv'

comparison_df.to_csv(comparison_output_path, index=False)
prediction_df.to_csv(predictions_output_path, index=False)

print(f"Métricas exportadas para: {comparison_output_path}")
print(f"Previsões exportadas para: {predictions_output_path}")